<a href="https://colab.research.google.com/github/ONESFA/PINNs-and-FDM-FEM/blob/main/notebooks/PINNS_inverse_pond.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 연못 역문제: 부표 센서만 보고 파속 $c$ 찾기 (PINN vs FDM)

연못($[0,1]^2$)에 돌 3개를 떨어뜨렸을 때 수면 높이 $u(x,y,t)$ 는 $u_{tt}=c^2(u_{xx}+u_{yy})$ 를 따릅니다.
**파속 $c$ 를 모르고**, 연못 곳곳의 **부표 36개가 기록한 수면 높이(노이즈 포함)** 만 있을 때 $c$ 와 전체 파동을 알아내는 **역문제**입니다.

| 경우 | 아는 것 | FDM | PINN |
|---|---|---|---|
| **A** 돌 위치를 앎 | 식의 형태, 초기조건, 경계조건, 센서 | $c$ 를 바꿔 가며 정문제를 반복 계산해 센서에 맞춤 | 한 번 학습하면서 $c$ 도 함께 학습 |
| **B** 돌 위치를 모름 | 식의 형태, 경계조건, 센서 | 초기조건이 없어 **계산을 시작할 수 없음** → 추측한 초기조건으로 돌리면 틀린 답 | 초기조건 손실을 빼고 센서 데이터로 대신함 |

미리 예상하면 이렇습니다. **A에서는 FDM이 계산 시간으로 이길 가능성이 큽니다.** 정문제 한 번이 1초도 안 걸리기 때문입니다.
**B에서는 FDM이 그대로는 풀 수 없습니다.** FDM으로 하려면 초기 수면 전체(격자점 약 1만 개)를 미지수로 두고 adjoint 최적화를 따로 구현해야 하지만, PINN은 손실 항 하나를 빼기만 하면 됩니다.
PINN의 장점은 계산 속도가 아니라 **정보가 부족해도 데이터로 메워서 풀 수 있다**는 데 있습니다.

돌 배치는 0번 설정의 `STONES_MODE` 로 고릅니다. `"fixed"` 는 정해 둔 3개, `"random"` 은 `STONE_SEED` 에 따라 무작위로 놓습니다(가장자리·서로에서 일정 거리 확보).

노트북 구성: 0 설정 → 1 연못 정답 만들기 → 2 신경망 → 3 손실 → 4 PINN 학습(A, B) → 5 FDM 역문제 → 6 비교 → 7 저장

In [ ]:
import time
import math
import copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ============================================
# 0. 실험 설정 — 하이퍼파라미터는 전부 이 셀에서만 바꾼다
# ============================================
SEED = 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# --- 연못 (정답: 학습에는 센서 데이터로만 들어감) ---
T = 1.0                     # 관측 시간 [0, T]
C_TRUE = 0.8                # 실제 파속 (PINN / FDM 은 모름)
STONES_MODE = "fixed"            # 돌 배치: "fixed" (아래 STONES_FIXED) | "random" (STONE_SEED 로 무작위)
STONES_FIXED = [(0.30, 0.35, 1.00),   # (x, y, 진폭)
                (0.68, 0.30, 0.70),
                (0.55, 0.72, 0.85)]
N_STONES = 3                     # random: 돌 개수
STONE_SEED = 1                   # random: 시드만 바꾸면 다른 배치
STONE_MARGIN = 0.25              # random: 가장자리에서 최소 거리 (초기 물결이 경계에서 ~1e-4 이하 -> u = 0 과 어긋나지 않음)
STONE_MIN_DIST = 0.2             # random: 돌끼리 최소 거리 (물결이 처음부터 한 덩어리로 뭉치지 않게)
STONE_AMP = (0.6, 1.0)           # random: 진폭 범위
SIGMA = 0.06                # 돌이 만든 초기 물결의 폭 (가우시안 표준편차)
FDM_TRUE_NX = 201           # '실제 연못' 을 만드는 고해상도 FDM 격자 (경계 포함 한 변 점 수)
FDM_CFL = 0.5               # c*dt/h  (2차원 안정 조건 c*dt/h <= 1/sqrt(2))

# --- 관측: 부표 센서 ---
N_SENSOR_SIDE = 6           # 6 x 6 = 36개, x, y = 0.1 ~ 0.9 등간격
N_T_OBS = 101               # 센서 기록 시각 수 (0 ~ T 등간격)
NOISE = 0.01                # 노이즈 표준편차 = NOISE * 최대 진폭

# --- PINN ---
C_INIT = 1.3                # c 의 시작 추정값
INPUT_NORM = True           # 입력 (x, y, t) -> [-1, 1]
HARD_BC = True              # u = 16 x(1-x) y(1-y) N  (연못 가장자리 u = 0)
WIDTH, DEPTH = 64, 5        # 은닉층 폭, 은닉층 수 (돌 여러 개의 뾰족한 물결을 표현하려고 v3 보다 크게)
FOURIER_SIGMA = 2.0         # > 0 이면 random Fourier feature 입력. 돌 3개의 뾰족한 물결 학습에 필요 (0 = 끔)
N_FOURIER = 64
N_f = 10000                 # PDE 내부 점
N_IC = 2000                 # 초기조건 점 (경우 A 에서만 사용)
RESAMPLE = True             # Adam 단계에서 매 epoch 내부 점 재샘플링

# 손실 가중치. PDE 잔차와 초기 속도는 크기가 1/sigma^2, 1/sigma 배라 3번 셀에서 미리 정규화한다
W_PDE, W_DATA, W_IC, W_ICV = 1.0, 1.0, 1.0, 1.0

# --- 최적화 ---
EPOCHS_ADAM = 15000
LR_START, LR_END = 1e-3, 1e-5    # 신경망 학습률 (지수 감쇠)
LR_C = 5e-3                      # log c 의 학습률 (같은 비율로 감쇠)
C_WARMUP = 2000                  # 처음 이 epoch 동안은 c 를 고정하고 파동장부터 센서에 맞춘다
USE_LBFGS = True
LBFGS_CHUNKS = 20
LBFGS_ITERS_PER_CHUNK = 50
PRINT_EVERY = 1000

# --- FDM 역문제 ---
FDM_INV_NX = 101            # 역문제용 FDM 격자 (정답과 다른 격자 -> inverse crime 방지)
C_SEARCH = (0.3, 2.0)       # c 탐색 구간
N_SCAN = 12                 # 먼저 등간격으로 훑는 c 개수 (지역 최솟값 회피)
GOLDEN_TOL = 1e-3           # 황금분할 탐색 종료 구간 폭
WRONG_STONES = [(0.50, 0.50, 1.00)]   # 경우 B: 돌 위치를 모를 때 FDM 이 쓰는 추측 ("가운데 하나")


def sync():
    """GPU 연산은 비동기이므로 시간 측정 직전에 동기화한다."""
    if device.type == "cuda":
        torch.cuda.synchronize()


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)


set_seed(SEED)


def make_stones():
    """STONES_MODE 에 따라 돌 배치를 정한다. random 이면 조건을 만족할 때까지 다시 뽑는다 (rejection sampling).
    학습 난수와 섞이지 않도록 STONE_SEED 전용 난수 생성기를 쓴다."""
    if STONES_MODE == "fixed":
        return list(STONES_FIXED)
    rng_s = np.random.default_rng(STONE_SEED)
    stones = []
    for _ in range(10000):
        x, y = rng_s.uniform(STONE_MARGIN, 1 - STONE_MARGIN, 2)
        if all(math.hypot(x - sx_, y - sy_) >= STONE_MIN_DIST for sx_, sy_, _ in stones):
            stones.append((round(float(x), 3), round(float(y), 3), round(float(rng_s.uniform(*STONE_AMP)), 3)))
            if len(stones) == N_STONES:
                return stones
    raise ValueError("조건을 만족하는 돌 배치를 못 찾음: N_STONES 를 줄이거나 STONE_MIN_DIST / STONE_MARGIN 을 줄이세요")


STONES = make_stones()
print(f"돌 배치 ({STONES_MODE}) (x, y, 진폭):", STONES)

# 1번: '실제 연못' 만들기와 부표 관측

정답 파동은 고해상도 FDM(`FDM_TRUE_NX`)으로 만듭니다. PINN과 FDM 역문제에는 **센서 기록만** 넘어갑니다.
FDM은 v3의 leapfrog를 초기조건 함수와 관측 시각을 받도록 일반화했습니다. 첫 단계는 $u^1=u^0+\Delta t\,v_0+\tfrac{\lambda^2}{2}\Delta_h u^0$ 이고, CFL 조건을 assert로 검사합니다.

In [ ]:
# ============================================
# 1. 돌 초기조건, 일반화 FDM, 정답 시뮬레이션, 센서 관측
# ============================================
def u0_stones(x, y, stones=STONES, sigma=SIGMA):
    """돌이 만든 초기 수면: 가우시안 합. numpy 배열 / torch 텐서 모두 가능."""
    lib = torch if torch.is_tensor(x) else np
    u = 0.0 * x
    for xs, ys, a in stones:
        u = u + a * lib.exp(-((x - xs) ** 2 + (y - ys) ** 2) / (2 * sigma ** 2))
    return u


def fdm_wave_general(u0_fn, nx, T_end, c, cfl=FDM_CFL, n_obs=N_T_OBS):
    """단위 정사각형, u = 0 경계, u(.,0) = u0_fn(X, Y), u_t(.,0) = 0.
    0 ~ T_end 를 n_obs 개 시각으로 나눈 시각마다 전체 격자를 저장. 반환: xs, ts, U[n_obs, nx, nx], info"""
    h = 1.0 / (nx - 1)
    m = n_obs - 1
    nt = math.ceil(T_end / (cfl * h / c))
    nt = m * math.ceil(nt / m)                    # 관측 시각이 정확히 격자 위에 오도록
    dt = T_end / nt
    lam2 = (c * dt / h) ** 2
    assert 2 * lam2 <= 1.0 + 1e-12, f"CFL 위반: c dt / h = {math.sqrt(lam2):.3f}"
    every = nt // m

    xs = np.linspace(0.0, 1.0, nx)
    X, Y = np.meshgrid(xs, xs, indexing="xy")      # U[k, iy, ix]

    def lap(u):
        return u[2:, 1:-1] + u[:-2, 1:-1] + u[1:-1, 2:] + u[1:-1, :-2] - 4.0 * u[1:-1, 1:-1]

    t0 = time.time()
    U = np.zeros((n_obs, nx, nx))
    u_prev = u0_fn(X, Y)
    u_prev[0, :] = u_prev[-1, :] = u_prev[:, 0] = u_prev[:, -1] = 0.0
    u_curr = np.zeros_like(u_prev)
    u_curr[1:-1, 1:-1] = u_prev[1:-1, 1:-1] + 0.5 * lam2 * lap(u_prev)   # v0 = 0
    U[0] = u_prev
    if every == 1:
        U[1] = u_curr
    u_next = np.zeros_like(u_prev)
    for n in range(1, nt):
        u_next[1:-1, 1:-1] = 2.0 * u_curr[1:-1, 1:-1] - u_prev[1:-1, 1:-1] + lam2 * lap(u_curr)
        u_prev, u_curr, u_next = u_curr, u_next, u_prev
        if (n + 1) % every == 0:
            U[(n + 1) // every] = u_curr
    info = dict(nx=nx, nt=nt, dt=dt, cfl=math.sqrt(lam2), time=time.time() - t0)
    return xs, np.linspace(0.0, T_end, n_obs), U, info


# --- 정답 시뮬레이션 ---
xs_true, ts_obs, U_true, info_true = fdm_wave_general(u0_stones, FDM_TRUE_NX, T, C_TRUE)
print(f"정답 FDM: nx={info_true['nx']}, nt={info_true['nt']}, c dt/h={info_true['cfl']:.3f}, {info_true['time']:.2f}s")

# --- 부표 센서: 두 FDM 격자 모두의 격자점 위에 놓이도록 0.1 ~ 0.9 등간격 ---
sensor_1d = np.linspace(0.1, 0.9, N_SENSOR_SIDE)
SX, SY = np.meshgrid(sensor_1d, sensor_1d, indexing="xy")
sx, sy = SX.ravel(), SY.ravel()                                   # (N_s,)
idx_true = (np.round(sy * (FDM_TRUE_NX - 1)).astype(int), np.round(sx * (FDM_TRUE_NX - 1)).astype(int))
idx_inv = (np.round(sy * (FDM_INV_NX - 1)).astype(int), np.round(sx * (FDM_INV_NX - 1)).astype(int))

rng = np.random.default_rng(SEED)
amp = np.abs(U_true[0]).max()
d_clean = U_true[:, idx_true[0], idx_true[1]]                     # (N_T_OBS, N_s)
d_obs = d_clean + NOISE * amp * rng.standard_normal(d_clean.shape)
print(f"센서 {len(sx)}개 x 시각 {N_T_OBS}개 = 관측값 {d_obs.size}개, 노이즈 표준편차 {NOISE * amp:.4f}")

# PINN 용 관측 텐서 (시각 x 센서 를 펼침)
TT, _ = np.meshgrid(ts_obs, np.arange(len(sx)), indexing="ij")
to_t = lambda a: torch.tensor(np.asarray(a, dtype=np.float32).reshape(-1, 1), device=device)
x_d, y_d = to_t(np.tile(sx, N_T_OBS)), to_t(np.tile(sy, N_T_OBS))
t_d, u_d = to_t(TT), to_t(d_obs)

# --- 연못 배치 그림 ---
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
X_np, Y_np = np.meshgrid(xs_true, xs_true, indexing="xy")
cf = ax[0].contourf(X_np, Y_np, U_true[0], levels=50, cmap="RdBu_r")
ax[0].scatter(sx, sy, s=18, c="k", marker="^", label="buoy sensor")
ax[0].scatter([s[0] for s in STONES], [s[1] for s in STONES], s=90, facecolors="none", edgecolors="lime", lw=2, label="stone")
ax[0].set_title("Pond at t = 0 (true), sensors and stones")
ax[0].set_aspect("equal")
ax[0].legend(loc="upper right", fontsize=8)
fig.colorbar(cf, ax=ax[0])
for j in [0, 14, 35]:
    ax[1].plot(ts_obs, d_obs[:, j], ".", ms=3, label=f"sensor {j} observed")
    ax[1].plot(ts_obs, d_clean[:, j], "-", lw=1, color=ax[1].lines[-1].get_color())
ax[1].set_title("Buoy records (dots: noisy observation, line: truth)")
ax[1].set_xlabel("t")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

# **2번 심층 신경망 근사**

v3 의 `PINN` (입력 정규화, hard BC) 에 폭·깊이 설정과 선택적 Fourier feature 입력을 더했습니다.
파속은 신경망 밖의 학습 파라미터 $\log c$ 로 두고 $c=e^{\log c}$ 로 써서 항상 양수가 되게 합니다 (4번 `train_inverse`).

In [ ]:
class PINN(nn.Module):

    def __init__(self, width=WIDTH, depth=DEPTH, normalize=INPUT_NORM, hard_bc=HARD_BC,
                 fourier_sigma=FOURIER_SIGMA, n_fourier=N_FOURIER):
        super().__init__()
        self.normalize = normalize
        self.hard_bc = hard_bc
        if fourier_sigma > 0:      # random Fourier feature: [sin(pi z B), cos(pi z B)]
            self.register_buffer("B", torch.randn(3, n_fourier) * fourier_sigma)
            n_in = 2 * n_fourier
        else:
            self.B = None
            n_in = 3               # (x, y, t)

        layers = [nn.Linear(n_in, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x, y, t):
        if self.normalize:         # [0,1] x [0,1] x [0,T]  ->  [-1,1]^3
            inputs = torch.cat([2 * x - 1, 2 * y - 1, 2 * t / T - 1], dim=1)
        else:
            inputs = torch.cat([x, y, t], dim=1)
        if self.B is not None:
            z = math.pi * inputs @ self.B
            inputs = torch.cat([torch.sin(z), torch.cos(z)], dim=1)
        out = self.net(inputs)
        if self.hard_bc:           # 연못 가장자리에서 u = 0
            out = 16.0 * x * (1 - x) * y * (1 - y) * out
        return out

# **3번 미분 연산 및 손실(Loss) 계산 모듈**

$$\mathcal L=w_{pde}\,\underbrace{\mathrm{mean}\big(\sigma^2 r\big)^2}_{\text{PDE}}+w_{data}\,\underbrace{\mathrm{mean}\big(u_\theta(x_s,y_s,t_k)-d_{sk}\big)^2}_{\text{센서}}
+\Big[w_{ic}\,\mathrm{mean}(u_\theta(\cdot,0)-u_0)^2+w_{icv}\,\mathrm{mean}(\sigma\,u_{\theta,t}(\cdot,0))^2\Big]_{\text{경우 A 에서만}}$$

- 잔차 $r=u_{tt}-c^2\Delta u$ 는 $u$ 의 $1/\sigma^2$ 배 크기($\sigma$ = 물결 폭)라 $\sigma^2$ 를 곱해 센서 항과 크기를 맞춥니다. 초기 속도 항도 같은 이유로 $\sigma$ 를 곱합니다.
- **경우 B 는 초기조건 항이 아예 없습니다.** 돌 위치에 대한 정보는 센서 기록과 파동방정식에서만 나옵니다.
- `pde_residual` 은 v3 와 같은 함수이며, `c` 에 학습 중인 텐서를 넘기면 $c$ 로도 기울기가 흐릅니다.

In [ ]:
# ============================================
# PDE Residual:  u_tt - c^2 (u_xx + u_yy)   (v3 와 동일, c 는 float 또는 텐서)
# ============================================
def pde_residual(model, x, y, t, c=1.0):
    x = x.clone().detach().requires_grad_(True)
    y = y.clone().detach().requires_grad_(True)
    t = t.clone().detach().requires_grad_(True)

    u = model(x, y, t)

    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_y = torch.autograd.grad(u, y, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]

    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_tt = torch.autograd.grad(u_t, t, grad_outputs=torch.ones_like(u_t), create_graph=True)[0]

    return u_tt - c**2 * (u_xx + u_yy)


# Initial Displacement / Velocity Loss (v3 와 동일)
def loss_ic(model, x_ic, y_ic, u_ic):
    t_ic = torch.zeros_like(x_ic)
    u_pred = model(x_ic, y_ic, t_ic)
    return torch.mean((u_pred - u_ic) ** 2)


def loss_ic_velocity(model, x_ic, y_ic, v_ic):
    t_ic = torch.zeros_like(x_ic)
    t_ic.requires_grad_(True)
    u_pred = model(x_ic, y_ic, t_ic)
    u_t = torch.autograd.grad(u_pred, t_ic, grad_outputs=torch.ones_like(u_pred), create_graph=True)[0]
    return torch.mean((u_t - v_ic) ** 2)


# 센서 데이터 손실
def loss_data(model):
    return torch.mean((model(x_d, y_d, t_d) - u_d) ** 2)


def make_collocation():
    """PDE 내부 점 (x_f, y_f, t_f) 와 초기조건 점 (x_ic, y_ic, u_ic, v_ic)."""
    xyz = torch.rand(N_f, 3, device=device)
    x_f, y_f, t_f = xyz[:, 0:1], xyz[:, 1:2], xyz[:, 2:3] * T
    xy = torch.rand(N_IC, 2, device=device)
    x_ic, y_ic = xy[:, 0:1], xy[:, 1:2]
    u_ic = u0_stones(x_ic, y_ic)
    return x_f, y_f, t_f, x_ic, y_ic, u_ic, torch.zeros_like(u_ic)


def inverse_loss(model, log_c, pts, use_ic):
    """반환: L_total, (L_pde, L_data, L_ic)."""
    x_f, y_f, t_f, x_ic, y_ic, u_ic, v_ic = pts
    c_hat = torch.exp(log_c)
    L_pde = torch.mean((SIGMA ** 2 * pde_residual(model, x_f, y_f, t_f, c_hat)) ** 2)
    L_data = loss_data(model)
    L = W_PDE * L_pde + W_DATA * L_data
    if use_ic:                                # 경우 A: 돌 위치(초기조건)를 앎
        L_ic = W_IC * loss_ic(model, x_ic, y_ic, u_ic) + W_ICV * SIGMA ** 2 * loss_ic_velocity(model, x_ic, y_ic, v_ic)
        L = L + L_ic
    else:                                     # 경우 B: 초기조건 항 없음
        L_ic = torch.zeros((), device=device)
    return L, (L_pde, L_data, L_ic)

# **4번 PINN 역문제 학습** (경우 A, B)

`train_inverse(use_ic)` 하나로 두 경우를 똑같이 학습합니다. Adam(신경망과 $\log c$ 의 학습률을 따로 둠) → L-BFGS 순서입니다.
처음 `C_WARMUP` epoch 동안은 $c$ 를 고정합니다. 파동장이 아직 센서에 안 맞는 상태에서 $c$ 를 같이 움직이면, PDE 손실을 쉽게 줄이려고 $c$ 가 0 쪽으로 빠지기 때문입니다.

L-BFGS 가 발산하면 손실이 가장 낮았던 상태로 되돌립니다(v3 와 같은 방식). 매 step 의 $c$ 를 기록해 수렴 과정을 그립니다.

In [ ]:
def train_inverse(use_ic, label):
    set_seed(SEED)                                    # 두 경우가 같은 초기 가중치에서 시작
    model = PINN().to(device)
    log_c = nn.Parameter(torch.tensor(math.log(C_INIT), device=device))

    optimizer = torch.optim.Adam([{"params": model.parameters(), "lr": LR_START},
                                  {"params": [log_c], "lr": LR_C}])
    gamma = (LR_END / LR_START) ** (1.0 / EPOCHS_ADAM)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

    hist = {"total": [], "pde": [], "data": [], "ic": [], "c": []}
    train_time = 0.0

    def record(L, parts):
        hist["total"].append(L.item())
        hist["pde"].append(parts[0].item())
        hist["data"].append(parts[1].item())
        hist["ic"].append(parts[2].item())
        hist["c"].append(math.exp(log_c.item()))

    print(f"===== PINN-{label} ({'초기조건 앎' if use_ic else '초기조건 모름'}) =====")
    pts = make_collocation()
    for epoch in range(EPOCHS_ADAM):
        t_start = time.time()
        if RESAMPLE:
            pts = make_collocation()
        log_c.requires_grad_(epoch >= C_WARMUP)      # 워밍업 동안 c 고정 (파동장이 틀린 채로 c 가 엉뚱하게 움직이는 것 방지)
        optimizer.zero_grad()
        L, parts = inverse_loss(model, log_c, pts, use_ic)
        L.backward()
        optimizer.step()
        scheduler.step()
        sync()
        train_time += time.time() - t_start
        record(L, parts)
        if epoch % PRINT_EVERY == 0 or epoch == EPOCHS_ADAM - 1:
            print(f"Epoch {epoch:5d} | Total {L.item():.3e} | PDE {parts[0].item():.3e} | "
                  f"Data {parts[1].item():.3e} | IC {parts[2].item():.3e} | c {hist['c'][-1]:.4f} | {train_time:.1f}s")
    adam_steps = len(hist["total"])

    if USE_LBFGS:
        log_c.requires_grad_(True)
        pts = make_collocation()                      # 고정 점
        lbfgs = torch.optim.LBFGS(list(model.parameters()) + [log_c], lr=1.0,
                                  max_iter=LBFGS_ITERS_PER_CHUNK, history_size=50,
                                  tolerance_grad=1e-10, tolerance_change=1e-14,
                                  line_search_fn="strong_wolfe")

        def closure():
            lbfgs.zero_grad()
            L, parts = inverse_loss(model, log_c, pts, use_ic)
            L.backward()
            record(L, parts)
            return L

        best = {"loss": float("inf"), "state": copy.deepcopy(model.state_dict()), "log_c": log_c.item()}
        for chunk in range(LBFGS_CHUNKS):
            t_start = time.time()
            lbfgs.step(closure)
            sync()
            train_time += time.time() - t_start
            if not math.isfinite(hist["total"][-1]):
                model.load_state_dict(best["state"])
                with torch.no_grad():
                    log_c.fill_(best["log_c"])
                print(f"L-BFGS 발산 -> 최적 상태로 복원 (loss {best['loss']:.3e})")
                break
            if hist["total"][-1] < best["loss"]:
                best = {"loss": hist["total"][-1], "state": copy.deepcopy(model.state_dict()), "log_c": log_c.item()}
            print(f"L-BFGS {chunk + 1:2d}/{LBFGS_CHUNKS} | Total {hist['total'][-1]:.3e} | c {math.exp(log_c.item()):.4f} | {train_time:.1f}s")

    c_hat = math.exp(log_c.item())
    print(f"PINN-{label}: c = {c_hat:.4f} (참값 {C_TRUE}, 오차 {abs(c_hat - C_TRUE) / C_TRUE:.2%}), 학습 {train_time:.1f}s\n")
    model.eval()
    return {"model": model, "c": c_hat, "hist": hist, "time": train_time, "adam_steps": adam_steps}


res_A = train_inverse(use_ic=True, label="A")
res_B = train_inverse(use_ic=False, label="B")

# **5번 FDM 역문제**

**경우 A (돌 위치를 앎):** FDM 은 $c$ 를 직접 학습할 수 없으므로, $c$ 를 정해 정문제를 풀고 센서 오차 $J(c)=\mathrm{mean}(u^{FDM}_c(x_s,t_k)-d_{sk})^2$ 를 재는 일을 반복합니다.
파동은 $c$ 가 조금만 달라도 위상이 한 주기 밀려 $J(c)$ 에 지역 최솟값이 생길 수 있습니다(cycle skipping). 그래서 먼저 `N_SCAN` 개 값을 훑고, 가장 좋은 값 주변에서 황금분할 탐색을 합니다.
**정문제를 몇 번 풀었는지**와 총 시간을 기록합니다.

**경우 B (돌 위치를 모름):** 초기조건이 없으면 FDM 은 한 걸음도 갈 수 없습니다. 비교를 위해 정보가 틀린 FDM 두 개를 돌립니다.
- *틀린 c*: 돌 위치는 맞고 $c$ 는 시작 추정값 `C_INIT`
- *틀린 초기조건*: $c$ 는 참값이지만 돌 위치를 `WRONG_STONES`("가운데 하나")로 추측

In [ ]:
# ============================================
# 5. FDM: (A) c 탐색, (B) 틀린 정보로 돌린 FDM
# ============================================
fdm_evals = []          # (c, J) — 정문제를 풀 때마다 기록


def fdm_misfit(c_val):
    _, _, U, _ = fdm_wave_general(u0_stones, FDM_INV_NX, T, c_val)
    J = float(np.mean((U[:, idx_inv[0], idx_inv[1]] - d_obs) ** 2))
    fdm_evals.append((c_val, J))
    return J


t_start = time.time()
# (1) 등간격 훑기
scan = np.linspace(*C_SEARCH, N_SCAN)
J_scan = [fdm_misfit(cv) for cv in scan]
k = int(np.argmin(J_scan))
lo, hi = scan[max(k - 1, 0)], scan[min(k + 1, N_SCAN - 1)]
# (2) 황금분할 탐색
g = (math.sqrt(5) - 1) / 2
a, b = lo + (1 - g) * (hi - lo), lo + g * (hi - lo)
Ja, Jb = fdm_misfit(a), fdm_misfit(b)
while hi - lo > GOLDEN_TOL:
    if Ja < Jb:
        hi, b, Jb = b, a, Ja
        a = lo + (1 - g) * (hi - lo)
        Ja = fdm_misfit(a)
    else:
        lo, a, Ja = a, b, Jb
        b = lo + g * (hi - lo)
        Jb = fdm_misfit(b)
c_fdm = 0.5 * (lo + hi)
fdm_search_time = time.time() - t_start
n_fdm_solves = len(fdm_evals)
print(f"FDM 탐색: c = {c_fdm:.4f} (참값 {C_TRUE}, 오차 {abs(c_fdm - C_TRUE) / C_TRUE:.2%}), "
      f"정문제 {n_fdm_solves}회, {fdm_search_time:.2f}s")

# 찾은 c 로 한 번 더 풀어 파동장 저장
_, _, U_fdm_A, _ = fdm_wave_general(u0_stones, FDM_INV_NX, T, c_fdm)

# (B) 정보가 틀린 FDM
_, _, U_wrong_c, info_wc = fdm_wave_general(u0_stones, FDM_INV_NX, T, C_INIT)
_, _, U_wrong_ic, info_wi = fdm_wave_general(lambda X, Y: u0_stones(X, Y, stones=WRONG_STONES), FDM_INV_NX, T, C_TRUE)
print(f"틀린 c FDM ({info_wc['time']:.3f}s), 틀린 초기조건 FDM ({info_wi['time']:.3f}s) 완료")

# **6번 비교**

모든 오차는 정답 FDM 과 같은 격자점(역문제 격자 101×101)과 시각 21개에서 계산한 전체 파동장의 상대 $L_2$ 오차입니다.

In [ ]:
# ============================================
# 6. 비교 표
# ============================================
STEP = (FDM_TRUE_NX - 1) // (FDM_INV_NX - 1)          # 정답 격자 -> 역문제 격자
T_SUB = slice(None, None, 5)                           # 시각 21개
U_ref = U_true[T_SUB, ::STEP, ::STEP]
xs_inv = np.linspace(0.0, 1.0, FDM_INV_NX)
ts_sub = ts_obs[T_SUB]


def pinn_field(model, ts, xs=xs_inv):
    """PINN 을 (시각, y, x) 격자에서 계산 -> numpy [len(ts), n, n]."""
    Xg, Yg = np.meshgrid(xs, xs, indexing="xy")
    xf, yf = to_t(Xg), to_t(Yg)
    out = []
    with torch.no_grad():
        for tv in ts:
            out.append(model(xf, yf, torch.full_like(xf, float(tv))).cpu().numpy().reshape(Xg.shape))
    return np.stack(out)


def rel_err(U):
    return np.linalg.norm(U - U_ref) / np.linalg.norm(U_ref)


U_pinn_A = pinn_field(res_A["model"], ts_sub)
U_pinn_B = pinn_field(res_B["model"], ts_sub)

rows = [
    ("FDM 탐색 (A)",        "PDE, IC, BC, 센서", f"정문제 {n_fdm_solves}회", fdm_search_time, c_fdm, rel_err(U_fdm_A[T_SUB])),
    ("PINN-A",              "PDE, IC, BC, 센서", "학습 1회",                  res_A["time"],   res_A["c"], rel_err(U_pinn_A)),
    ("PINN-B",              "PDE, BC, 센서",     "학습 1회",                  res_B["time"],   res_B["c"], rel_err(U_pinn_B)),
    ("FDM, 틀린 c",         "PDE, IC, BC",       "정문제 1회",                info_wc["time"], C_INIT,     rel_err(U_wrong_c[T_SUB])),
    ("FDM, 틀린 초기조건",  "PDE, BC, 참값 c",   "정문제 1회",                info_wi["time"], C_TRUE,     rel_err(U_wrong_ic[T_SUB])),
]
print(f"{'방법':<16}{'필요한 정보':<18}{'계산':<12}{'시간[s]':>9}{'c':>9}{'c 오차':>9}{'파동장 relL2':>14}")
for name, info, cost, sec, cv, err in rows:
    c_note = "(고정)" if name.startswith("FDM,") else f"{abs(cv - C_TRUE) / C_TRUE:.2%}"
    print(f"{name:<16}{info:<18}{cost:<12}{sec:9.2f}{cv:9.4f}{c_note:>9}{err:14.3e}")
print("\n'틀린 초기조건' FDM 은 c 를 참값으로 줬는데도 돌 위치를 몰라 파동장이 틀린다. 경우 B 에서 FDM 이 할 수 있는 최선이 이 줄이다.")

In [ ]:
# 그림 1: c 수렴 과정
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [2, 1]})
for res, lab in [(res_A, "PINN-A (IC known)"), (res_B, "PINN-B (IC unknown)")]:
    ax[0].plot(np.arange(1, len(res["hist"]["c"]) + 1), res["hist"]["c"], lw=1.2, label=lab)
ax[0].axvline(res_A["adam_steps"], ls="--", color="gray", lw=1)
ax[0].axhline(C_TRUE, color="k", ls=":", label=f"true c = {C_TRUE}")
ax[0].set_xlabel("step (Adam epoch / L-BFGS eval)")
ax[0].set_ylabel("c")
ax[0].set_title("PINN: c is learned during one training run")
ax[0].legend()

cs, Js = zip(*fdm_evals)
ax[1].semilogy(cs[:N_SCAN], Js[:N_SCAN], "o", color="tab:gray", label="scan")
ax[1].semilogy(cs[N_SCAN:], Js[N_SCAN:], ".", color="tab:red", label="golden-section")
ax[1].axvline(C_TRUE, color="k", ls=":")
ax[1].set_xlabel("c")
ax[1].set_title(f"FDM: misfit J(c), {n_fdm_solves} forward solves")
ax[1].legend()
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 그림 2: 수면 스냅샷 — 정답 / PINN-B / 틀린 초기조건 FDM
snap_k = [0, len(ts_sub) // 2, len(ts_sub) - 1]
Xi, Yi = np.meshgrid(xs_inv, xs_inv, indexing="xy")
rows_img = [("Truth", U_ref), ("PINN-B (IC unknown)", U_pinn_B), ("FDM, wrong IC", U_wrong_ic[T_SUB])]
fig, axes = plt.subplots(3, 3, figsize=(14, 13))
for j, kk in enumerate(snap_k):
    vmax = np.abs(U_ref[kk]).max()
    for i, (name, U) in enumerate(rows_img):
        cf = axes[i, j].contourf(Xi, Yi, U[kk], levels=np.linspace(-vmax, vmax, 41), cmap="RdBu_r", extend="both")
        axes[i, j].set_title(f"{name}, t = {ts_sub[kk]:.2f}")
        axes[i, j].set_aspect("equal")
        fig.colorbar(cf, ax=axes[i, j], shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# 그림 3: PINN-B 가 센서만 보고 복원한 초기 수면 (돌 위치를 찾았는가)
U0_B = U_pinn_B[0]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
vmax = np.abs(U_ref[0]).max()
for a, (name, U) in zip(ax[:2], [("True initial surface", U_ref[0]), ("PINN-B reconstruction, t = 0", U0_B)]):
    cf = a.contourf(Xi, Yi, U, levels=np.linspace(-vmax, vmax, 41), cmap="RdBu_r", extend="both")
    a.scatter([s[0] for s in STONES], [s[1] for s in STONES], s=120, facecolors="none", edgecolors="lime", lw=2)
    a.scatter(sx, sy, s=10, c="k", marker="^")
    a.set_title(name)
    a.set_aspect("equal")
    fig.colorbar(cf, ax=a, shrink=0.85)
cf = ax[2].contourf(Xi, Yi, np.abs(U0_B - U_ref[0]), levels=40, cmap="magma")
ax[2].set_title(f"|error| at t = 0  (rel L2 = {np.linalg.norm(U0_B - U_ref[0]) / np.linalg.norm(U_ref[0]):.2e})")
ax[2].set_aspect("equal")
fig.colorbar(cf, ax=ax[2], shrink=0.85)
plt.tight_layout()
plt.show()

In [ ]:
# 그림 4: 센서 시계열 — 관측 / 정답 / PINN-B / 틀린 정보 FDM
show = [0, 21]                         # 그릴 센서 번호 (0 ~ 35)
fig, ax = plt.subplots(1, len(show), figsize=(15, 4.2))
for a, j in zip(ax, show):
    with torch.no_grad():
        pB = res_B["model"](to_t(np.full_like(ts_obs, sx[j])), to_t(np.full_like(ts_obs, sy[j])), to_t(ts_obs)).cpu().numpy().ravel()
    a.plot(ts_obs, d_obs[:, j], ".", ms=4, color="gray", label="observed (noisy)")
    a.plot(ts_obs, d_clean[:, j], "k-", lw=1.5, label="truth")
    a.plot(ts_obs, pB, "-", lw=1.5, label="PINN-B")
    a.plot(ts_obs, U_wrong_ic[:, idx_inv[0][j], idx_inv[1][j]], "--", lw=1.2, label="FDM, wrong IC")
    a.plot(ts_obs, U_wrong_c[:, idx_inv[0][j], idx_inv[1][j]], ":", lw=1.2, label="FDM, wrong c")
    a.set_title(f"sensor {j} at ({sx[j]:.2f}, {sy[j]:.2f})")
    a.set_xlabel("t")
    a.grid(alpha=0.3)
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

### 결과 읽는 법

- **경우 A:** FDM 탐색과 PINN-A 모두 $c$ 를 찾아야 정상입니다. 걸린 시간은 FDM 이 훨씬 짧을 가능성이 큽니다. 2차원·단순 영역에서는 정문제 한 번이 매우 싸기 때문입니다. 이 결과는 숨기지 않고 그대로 보고합니다.
- **경우 B:** FDM 은 초기조건이 없으면 시작할 수 없고, 추측한 초기조건으로 돌리면 $c$ 가 정확해도 파동장이 크게 틀립니다(표의 마지막 줄).
  FDM 으로 이 문제를 풀려면 초기 수면 전체(101×101 ≈ 1만 개 값)를 미지수로 두고 adjoint 방정식을 유도·구현해 최적화해야 합니다.
  PINN 은 초기조건 손실 항을 **빼기만** 하고 같은 코드로 $c$ 와 전체 파동, 그리고 그림 3 처럼 돌의 위치까지 복원합니다.
- **무작위 배치일 때:** 돌이 가운데 근처에 떨어지면, 가운데에 하나를 추측한 `WRONG_STONES` FDM 이 우연히 덜 틀릴 수 있습니다. 표의 마지막 줄을 볼 때 돌 배치를 같이 확인하세요.
- **결론:** PINN 의 이점은 "같은 문제를 더 빨리 푼다"가 아니라 "정보가 빠진 문제를 데이터로 메워 푼다"는 데 있습니다. 센서 수(`N_SENSOR_SIDE`), 노이즈(`NOISE`), 돌의 폭(`SIGMA`)을 바꿔 PINN-B 가 언제까지 버티는지 보는 것이 다음 실험으로 좋습니다.
- **PINN 이 잘 안 될 때:** 센서 손실(Data)이 노이즈 수준 $(\text{NOISE}\cdot\text{진폭})^2\approx10^{-4}$ 까지 내려가는지 먼저 보세요. 안 내려가면 파동장을 못 배운 것이고, 그 상태의 $c$ 는 믿을 수 없습니다. 학습 횟수, `FOURIER_SIGMA`(1~4), `WIDTH/DEPTH`, `C_WARMUP` 순서로 조정해 보세요. `FOURIER_SIGMA=0` 이면 c 가 0.3~0.6 근처로 잘못 수렴하는 것을 확인했습니다.

# 7번 저장

In [ ]:
config = {k: v for k, v in globals().items()
          if k.isupper() and not k.startswith("_") and isinstance(v, (bool, int, float, str, list, tuple, dict))}
config.update(N_f=N_f, N_IC=N_IC, T=T)
torch.save({"model_A": res_A["model"].state_dict(), "model_B": res_B["model"].state_dict(),
            "c_A": res_A["c"], "c_B": res_B["c"], "c_fdm": c_fdm, "fdm_evals": fdm_evals,
            "hist_A": res_A["hist"], "hist_B": res_B["hist"], "config": config}, "pinn_inverse_pond.pt")
print("saved: pinn_inverse_pond.pt")